# Evo Kriging Task

The purpose of this notebook is to test the Evo Kriging task by performing an estimation and ultimately saving the results to blocksync such that they can be viewed in leapfrog.

---

First import necessary packages

In [ ]:
import pyarrow as pa
from dotenv import dotenv_values
from evo.blockmodels import BlockModelAPIClient
from evo.blockmodels.endpoints.models import ColumnHeaderType
from evo.widgets import FeedbackWidget, ServiceManagerWidget
from evo.objects import ObjectAPIClient

from evo.compute import JobClient

## Authentication

Next login to Seequent Evo and select an instance and workspace

In [ ]:
# autheticate and create service manager

evo_config = dotenv_values(".evo.env")
manager = await ServiceManagerWidget.with_auth_code(**evo_config).login()

## Establish Object API client
Now, establish a connection to the geoscience objects API

In [ ]:
# connect to Object API
environment = manager.get_environment()
connector = manager.get_connector()

object_client = ObjectAPIClient(environment, connector)
service_health = await object_client.get_service_health()

print(f"Object API is {service_health.status.name.lower()}")

data_client = object_client.get_data_client(manager.cache)

### Select Objects for Kriging
First, list all of the objects present in the workspace such that the desired objects can be selected as input for kriging

In [ ]:
# list objects in the workspace

offset = 0
while True:
    page = await object_client.list_objects(offset=offset, limit=10)
    if offset == 0:
        print(f"Found {page.total} object{'' if page.total == 1 else 's'}")
    for object in page:
        print(f"{object.path}: <{object.schema_id}> ({object.id})")

    if page.is_last:
        break
    else:
        offset = page.next_offset

 ### Download the desired objects

 Select and download the objects needed for kriging

In [ ]:
# download needed objects

comps = await object_client.download_object_by_path("Smokey_Resource: comps_10m.json")
variogram_model = await object_client.download_object_by_path(
    "Au_ppm in Smokey - INTR2: Transformed Variogram Model.json"
)
block_model = await object_client.download_object_by_path("Resource Model.json")
mgrid = await object_client.download_object_by_path("INTR2-testevokriging.json")

## Perform the Kriging

Setup the kriging parameters and start the kriging

In [ ]:
from evo.compute.endpoints.api import TasksApi

In [ ]:
tasks_api = TasksApi(connector)

In [ ]:
# setup kriging parameters

kriging = await JobClient.submit(
    connector=connector,
    org_id=environment.org_id,
    topic="geostatistics",
    task="kriging",
    parameters={
        "source": {
            "object": comps.metadata.url,
            "attribute": "attributes[?name=='Au_ppm']",
        },
        "target": {
            "object": mgrid.metadata.url,
            "attribute": {"operation": "create", "name": "Au_ok-test-eric1"},
        },
        "kriging_method": {"type": "ordinary"},
        "variogram": variogram_model.metadata.url,
        "neighborhood": {
            "ellipsoid": {
                "ellipsoid_ranges": {"major": 50.0, "semi_major": 50.0, "minor": 50.0},
                "rotation": {"dip_azimuth": 0.0, "dip": 0.0, "pitch": 0.0},
            },
            "max_samples": 40,
        },
    },
    result_type=dict,
    preview=True,
)


print(kriging)

Check the status of the kriging until finished

In [ ]:
# check the progress - check periodically until finished
response = await kriging.get_status()
print(response)

In [ ]:
# check the progress - check periodically until finished
response = await kriging.get_status()
print(response)

Get the kriging results

In [ ]:
# get the results when finished
await kriging.get_results()

## Retrieve the results
Get the updated masked grid 

In [ ]:
# reload the modified block model with kriging results
mgrid = await object_client.download_object_by_path("INTR2-testevokriging.json")

Now, access the kriged estimates 

In [ ]:
# retrieve the kriged values as a DataFrame
au_ok = await mgrid.download_dataframe(
    mgrid.search("cell_attributes[?name=='Au_ok2'].values")[0], fb=FeedbackWidget("Downloading DataFrame")
)

## Send results to Blocksync
Now get the blocksync model

In [ ]:
# now need to access the blocksync model - now need to move the data from the `mgrid` geoscience objec to blocksync model
bms_client = BlockModelAPIClient(environment, connector, cache=manager.cache)

bm_uuid = "961a391f-d0d6-4627-80b2-eef62a324a93"  # had to query from portal url

Get the coordinates and categorical data to associate values with correct domain

In [ ]:
# get the blockmodel coordinates and categories
bm = await bms_client.query_block_model_as_table(
    bm_id=bm_uuid, columns=["x", "y", "z", "GM"], column_headers=ColumnHeaderType.name
)
bm_df = bm.to_pandas()

Select a subset for only the domain of interest

In [ ]:
# select only INTR2 blocks (this was the area where mask=True)
intr2_df = bm_df[bm_df["GM"] == "INTR2"].copy()

Sort the coordinates such that the are correctly associated with the values

In [ ]:
# need to attach the kriged values to the intr2_df BUT the sort order isn't right.
# one way to handle this is to sort the coordinates before attaching to the kriged values
# but I havent been able to get it right :(
intr2_df.sort_values(by=["z", "x", "y"], inplace=True)

Give the column a name

In [ ]:
# give the new column a name - annoyingly cant update the same column so each time I createa new one (hence the incrementing numbers)
col_name = "Au_ok-test"

Add the values to the dataframe with sorted coordinates

In [ ]:
# attach grade values to the [sorted] intr2_df
intr2_df[col_name] = au_ok.values

Convert from pandas to pyarrow table to satisfy Blocksync

In [ ]:
# convert to a pyarrow table for upload to blocksync
intr2_table = pa.Table.from_pandas(intr2_df[["x", "y", "z", col_name]], preserve_index=False)
intr2_table

Upload to Blocksync

In [ ]:
# upload to blocksync
version_with_new_columns = await bms_client.add_new_columns(
    bm_id=bm_uuid,
    data=intr2_table,
)

In [ ]:
version_with_new_columns